In [1]:
from nbtools import NotebookCache, show

# results and prints live on disk, so they survive kernel restarts; one folder per notebook.
# No also= to the other flow-matching notebooks: their tests have the same code as ours
cache, clear_cache = NotebookCache('../cache/flow_matching_dit').split()

# Flow matching with a DiT velocity net

Third in the series, same data, loss, sampler and tests; only the velocity net changes. At 10k steps:

| velocity net                           | params | val gripper | same start, 8 / 16 steps per plan    |
| -------------------------------------- | ------ | ----------- | ------------------------------------ |
| MLP (`flow_matching_namespace.ipynb`)  | 867k   | 0.091       | 5/20, 17/20 (9 right / 8 left)       |
| 1-D U-Net (`flow_matching_unet.ipynb`) | 3.85M  | 0.035       | 20/20 (14 L / 6 R), 20/20 (all left) |

Here: a small **DiT** (Peebles & Xie 2023), the head current action experts use and the repo's flow-matching policy: one token per chunk step, self-attention across the 16 steps, the condition entering every block through adaLN-zero. Built one box at a time.

### step 0 — setup, copied from `flow_matching_unet.ipynb` step 0

T2-fixed: two mirrored bins, the cube always at the table centre, the expert flips a coin for the bin. 100 demos, 10 held out (5 per bin). Chunks of K = 16, everything on the GPU, normalisation stats, BC's image encoder.


In [2]:
import json
from pathlib import Path
from types import SimpleNamespace as namespace

import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from numpy.typing import NDArray

np.set_printoptions(precision=3, suppress=True)
REPO = Path.cwd().parent
ROOT = REPO / 'data/robosuite/pick_place_two_bins_fixed_noise0025_n100'

In [3]:
@cache
def load_table():
    # straight from the table, no video decode
    ds = LeRobotDataset('dream_robot/pick_place_two_bins', root=ROOT)
    return (
        np.asarray(ds.hf_dataset['observation.state'], dtype=np.float32),  # state [N, 8]
        np.asarray(ds.hf_dataset['action'], dtype=np.float32),  # action [N, 8]
        np.asarray(ds.hf_dataset['episode_index'], dtype=np.int64),  # episode [N], which demo
    )

load_table: cached, 1 call(s) stored


In [4]:
state, action, episode = load_table()  # three arrays, unpacked once: a tuple

# T2-fixed: the cube always starts at the table centre, so the scene never says which bin.
# The expert's coin isn't in the dataset; record.py logged it per recording attempt
attempts = [a for a in json.load(open(ROOT / 'bin_choices.json'))['attempts'] if a['kept']]
bin_of = np.array([a['target_bin'] for a in attempts])  # [100], episode e = the e-th kept attempt

# hold out 10 whole demos, 5 per bin: a plain random 10 came out 9 right / 1 left
rng = np.random.default_rng(0)
val_eps = np.concatenate(
    [rng.choice(np.flatnonzero(bin_of == b), 5, replace=False) for b in ['left', 'right']]
)
is_val = np.isin(episode, val_eps)  # [N]
show(
    state.shape,
    episode.max() + 1,
    bin_of.shape,
    (bin_of == 'left').sum(),
    (bin_of == 'right').sum(),
)
show(
    val_eps,
    (~is_val).sum(),
    is_val.sum(),
)

state.shape = (21741, 8)
episode.max() + 1 = np.int64(100)
bin_of.shape = (100,)
(bin_of == 'left').sum() = np.int64(50)
(bin_of == 'right').sum() = np.int64(50)
val_eps = array([59, 49, 27, 31, 79, 74, 48, 60, 87, 99])
(~is_val).sum() = np.int64(19555)
is_val.sum() = np.int64(2186)


In [5]:
import torch
from torch import Tensor

In [6]:
@cache
def load_images():
    # every frame decoded from both videos once, kept as uint8 (~2.1 GB)
    ds = LeRobotDataset('dream_robot/pick_place_two_bins', root=ROOT)
    top = np.zeros((len(ds), 3, 128, 128), dtype=np.uint8)  # [N, 3, H, W]
    wrist = np.zeros_like(top)
    for i in range(len(ds)):
        item = ds[i]
        top[i] = (item['observation.images.top'] * 255).round().to(torch.uint8).numpy()
        wrist[i] = (item['observation.images.wrist'] * 255).round().to(torch.uint8).numpy()
    return top, wrist

load_images: cached, 1 call(s) stored


In [7]:
top, wrist = load_images()  # two arrays, unpacked at once: a tuple
show(
    top.shape,
    wrist.shape,
    f'{(top.nbytes + wrist.nbytes) / 1e9:.1f} GB',
)

top.shape = (21741, 3, 128, 128)
wrist.shape = (21741, 3, 128, 128)
2.1 GB


In [8]:
def make_chunks(action: NDArray[np.float32], episode: NDArray[np.int64], K: int):
    # demo e's last row is where its run of rows ends, since demos are stored in order
    assert (np.diff(episode) >= 0).all(), 'demos not stored in order'
    last = np.searchsorted(episode, episode, side='right') - 1  # [N]
    idx = np.arange(len(action))[:, None] + np.arange(K)  # [N, K], rows t .. t+K-1
    pad = idx > last[:, None]  # [N, K], True where the chunk runs past its demo's end
    chunks = action[np.minimum(idx, last[:, None])]  # [N, K, 8], past the end: the final action
    return chunks, pad


K = 16  # chunk length, the repo's flow-matching value: 16 steps = 0.5 s (ACT used 50)
chunks, pad = make_chunks(action, episode, K)
show(
    chunks.shape,
    pad.any(1).sum(),
)

chunks.shape = (21741, 16, 8)
pad.any(1).sum() = np.int64(1500)


In [9]:
dev = 'cuda'


def split(x: NDArray) -> namespace:
    # train/val by episode, each on the GPU; images stay uint8 until a batch is drawn
    return namespace(train=torch.tensor(x[~is_val]).to(dev), val=torch.tensor(x[is_val]).to(dev))


def mean_std(x: Tensor, dims: tuple[int, ...]) -> namespace:
    # keepdim leaves a 1 for each reduced dim, so the stats broadcast straight back over x
    return namespace(mean=x.mean(dims, keepdim=True), std=x.std(dims, keepdim=True))


data = namespace(
    top=split(top),
    wrist=split(wrist),
    state=split(state),
    chunks=split(chunks),
    pad=split(pad),
)

# from train only, as bc_2 / act. Action stats from single actions: a chunk is made of the same
# actions, so they apply to every step of it. State and action [1, 8]; images [1, 3, 1, 1],
# from every 20th frame
stats = namespace(
    state=mean_std(data.state.train, (0,)),
    action=mean_std(data.chunks.train[:, 0], (0,)),
    top=mean_std(data.top.train[::20].float() / 255, (0, 2, 3)),
    wrist=mean_std(data.wrist.train[::20].float() / 255, (0, 2, 3)),
)
show(
    data.chunks.train.shape,
    data.chunks.val.shape,
    stats.action.std,
)
show(f'GPU {torch.cuda.memory_allocated() / 1e9:.1f} GB')

data.chunks.train.shape = torch.Size([19555, 16, 8])
data.chunks.val.shape = torch.Size([2186, 16, 8])
stats.action.std =
tensor([[0.1861, 0.1934, 0.0519, 0.1271, 0.0828, 0.1632, 0.2331, 0.4805]],
       device='cuda:0')
GPU 2.1 GB


In [10]:
from torch import nn


# bc_2's image encoder, unchanged (and act step 5): conv stack (bc_2 step 9) without its last
# ReLU, then spatial softmax (bc_2 step 15)
def make_conv() -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(3, 32, 5, 2, 2),  # -> [B, 32, 64, 64]
        nn.ReLU(),
        nn.Conv2d(32, 64, 3, 2, 1),  # -> [B, 64, 32, 32]
        nn.ReLU(),
        nn.Conv2d(64, 128, 3, 2, 1),  # -> [B, 128, 16, 16]
        nn.ReLU(),
        nn.Conv2d(128, 64, 3, 1, 1),  # -> [B, 64, 16, 16], no ReLU: the softmax wants negatives
    )


class SpatialSoftmax(nn.Module):
    cell_x: Tensor  # buffers, declared so the checker doesn't read them as Tensor | Module
    cell_y: Tensor

    def __init__(self, h: int, w: int):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(()))  # learned temperature, T = exp(log_t) > 0
        self.register_buffer('cell_x', torch.linspace(-1, 1, w).repeat(h))  # [H*W]
        self.register_buffer('cell_y', torch.linspace(-1, 1, h).repeat_interleave(w))  # [H*W]

    def forward(self, f: Tensor) -> Tensor:
        # f: [B, C, H, W] feature maps
        # out: [B, 2C], an (x, y) per map
        B, C, H, W = f.shape
        p = torch.softmax(f.reshape(B, C, H * W) / self.log_t.exp(), dim=-1)  # [B, C, H*W]
        return torch.stack([p @ self.cell_x, p @ self.cell_y], dim=-1).reshape(B, 2 * C)

### step 1 — tokens: one vector per chunk step, plus where it sits

A transformer works on a sequence of vectors. The chunk already is one: 16 steps × 8 numbers. Each step is lifted to `d` numbers by the same linear layer, then a learned position vector is added so step 3 and step 12 look different:

```
chunk   [B, K, 8]
tokens  = Linear(8 → d)(chunk) + pos          pos: [K, d], learned, one row per step
        -> [B, K, d]                          d = 128
```

The same `Linear` at every step is the U-Net's 1×1 conv: a kernel that sees one step at a time. Unlike the U-Net's kernel-5 convs, nothing here mixes steps yet; that is attention's job (step 2), and attention by itself can't tell steps apart, hence `pos`.


In [11]:
d = 128  # token width
tok = nn.Linear(8, d).to(dev)
pos = nn.Parameter(torch.randn(K, d, device=dev) * 0.02)  # small at init, learned
x1 = (data.chunks.train[:4] - stats.action.mean) / stats.action.std  # [4, K, 8], std units
h = tok(x1) + pos  # [4, K, 8] -> [4, K, d]; pos [K, d] broadcasts over the batch
show(
    x1.shape,
    h.shape,
    pos.shape,
)

x1.shape = torch.Size([4, 16, 8])
h.shape = torch.Size([4, 16, 128])
pos.shape = torch.Size([16, 128])


The per-step `Linear` is exactly a 1×1 `Conv1d` on the channels × time layout, the U-Net's view of the same chunk:


In [12]:
@lambda f: f()
def probe():
    conv = nn.Conv1d(8, d, kernel_size=1).to(dev)
    with torch.no_grad():
        conv.weight.copy_(tok.weight[:, :, None])  # [d, 8] -> [d, 8, 1]
        conv.bias.copy_(tok.bias)
        as_conv = conv(x1.transpose(1, 2)).transpose(1, 2)  # [4, K, d]
        show((as_conv - tok(x1)).abs().max())

(as_conv - tok(x1)).abs().max() = tensor(2.3842e-07, device='cuda:0')


### step 2 — self-attention: every step reads every other step

The U-Net mixed steps through kernel-5 convs: each step saw its ±2 neighbours, and the whole chunk only at the bottom of the U. Attention mixes all 16 at once, with weights computed from the tokens themselves:

```
h          [B, K, d]                       the tokens
q, k, v  = h Wq, h Wk, h Wv                [B, K, d] each: what I look for / what I offer / what I pass on
scores   = q kᵀ / √d                       [B, K, K]: score[i, j] = how much step i wants step j
A        = softmax over j (scores)         each row sums to 1
out      = (A v) Wo                        [B, K, d]: step i gets a weighted average of every v_j
```

**Multi-head:** split `d` into `H` heads of `d/H` numbers, run the above in each with its own `A`, concatenate. Different heads can look at different steps. `√(d/H)` then scales each head's scores, which keeps them from growing with the width (a dot product of `n` unit-variance numbers has variance `n`).


In [13]:
from torch import Tensor


def naive_attention(h: Tensor, mha: nn.MultiheadAttention) -> Tensor:
    # h: [B, L, d], the same weights as mha (its packed in_proj and out_proj)
    # out: [B, L, d], what mha(h, h, h) computes, written out
    B, L, D = h.shape
    H = mha.num_heads
    Wq, Wk, Wv = mha.in_proj_weight.chunk(3)  # [d, d] each
    bq, bk, bv = mha.in_proj_bias.chunk(3)  # [d] each
    q, k, v = h @ Wq.T + bq, h @ Wk.T + bk, h @ Wv.T + bv  # [B, L, d] each

    def heads(x: Tensor) -> Tensor:  # [B, L, d] -> [B, H, L, d/H]
        return x.reshape(B, L, H, D // H).transpose(1, 2)

    q, k, v = heads(q), heads(k), heads(v)
    scores = q @ k.transpose(-1, -2) / (D // H) ** 0.5  # [B, H, L, L]
    A = scores.softmax(dim=-1)  # each row sums to 1
    out = (
        (A @ v).transpose(1, 2).reshape(B, L, D)
    )  # [B, H, L, d/H] -> [B, L, d], heads side by side
    return mha.out_proj(out)

In [14]:
mha = nn.MultiheadAttention(d, num_heads=4, batch_first=True).to(dev)
with torch.no_grad():
    theirs, A = mha(h, h, h)  # A: [B, L, L], the attention averaged over heads
    ours = naive_attention(h, mha)
show(
    ours.shape,
    (ours - theirs).abs().max(),
    A.shape,
    A[0].sum(-1),
)

ours.shape = torch.Size([4, 16, 128])
(ours - theirs).abs().max() = tensor(1.7881e-07, device='cuda:0')
A.shape = torch.Size([4, 16, 16])
A[0].sum(-1) =
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       device='cuda:0')


Attention doesn't know order. Shuffle the 16 steps: without `pos`, the output is the same rows, shuffled the same way (it treats the chunk as a set). With `pos` added, the shuffled steps carry their original positions, and the output changes. That's why step 1 adds `pos`.


In [15]:
@lambda f: f()
def probe():
    perm = torch.randperm(K, device=dev)  # a shuffle of the 16 steps
    with torch.no_grad():
        no_pos = tok(x1)  # [4, K, d], tokens without position
        out = naive_attention(no_pos, mha)
        out_shuffled = naive_attention(no_pos[:, perm], mha)
        show(
            'no pos: shuffle in -> same rows shuffled out',
            (out_shuffled - out[:, perm]).abs().max(),
        )

        with_pos = tok(x1[:, perm]) + pos  # the steps shuffled, positions still 0..15
        out_pos = naive_attention(tok(x1) + pos, mha)
        show(
            'with pos: no longer just shuffled',
            (naive_attention(with_pos, mha) - out_pos[:, perm]).abs().max(),
        )

no pos: shuffle in -> same rows shuffled out
(out_shuffled - out[:, perm]).abs().max() = tensor(1.1921e-07, device='cuda:0')
with pos: no longer just shuffled
(naive_attention(with_pos, mha) - out_pos[:, perm]).abs().max() = tensor(0.0006, device='cuda:0')


**Reading it:** without `pos`, shuffled in = the same rows shuffled out, to 1e-7: attention alone sees a set. With `pos` the order shows, but only by 0.0006, because `pos` starts at ×0.02 the size of the tokens. Training grows it into whatever position code the task needs.


### step 3 — the transformer block: attention, then an MLP, each as a residual

```
x  = x + Attn(LN(x))          steps talk to each other
x  = x + MLP(LN(x))           each step thinks on its own: Linear(d → 4d), GELU, Linear(4d → d)
LN(z) = (z − mean(z)) / std(z) over each token's d numbers, then a learned scale and shift
```

The U-Net's residual block again (`x + f(x)`, step 2 there), with attention for the conv. **Pre-norm**: LN goes inside the branch, so the residual stream `x` is never normalised and passes straight through every block. LayerNorm is GroupNorm with one group per token, normalising each step's own `d` numbers. GELU is another smooth ReLU (`z · Φ(z)`, Φ the normal CDF), the transformer default.


In [21]:
import torch.nn.functional as F


class Attention(nn.Module):
    # step 2's formula, one line per stage; the fused kernel does softmax(q kᵀ / √(d/H)) v
    def __init__(self, d: int, heads: int = 4):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(d, 3 * d)  # Wq, Wk, Wv side by side
        self.out = nn.Linear(d, d)  # Wo

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, L, d]
        # out: [B, L, d]
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)  # [B, L, d] each
        q, k, v = (t.reshape(B, L, self.heads, D // self.heads).transpose(1, 2) for t in (q, k, v))
        o = F.scaled_dot_product_attention(q, k, v)  # [B, H, L, d/H]
        return self.out(o.transpose(1, 2).reshape(B, L, D))  # heads side by side, then Wo


class Block(nn.Module):
    # pre-norm transformer block; no condition yet (adaLN-zero is step 4)
    def __init__(self, d: int, heads: int = 4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attn = Attention(d, heads)
        self.norm2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, L, d]
        # out: [B, L, d]
        x = x + self.attn(self.norm1(x))  # steps talk to each other
        return x + self.mlp(self.norm2(x))  # each step on its own

`Attention` with step 2's `mha` weights copied in gives what `nn.MultiheadAttention` gives: the same q, k, v, heads and Wo, with the fused kernel in the middle (~330 µs fwd+bwd vs MHA's ~380 and the hand-written 530, at batch 64).


In [22]:
@lambda f: f()
def probe():
    att = Attention(d, heads=4).to(dev)
    with torch.no_grad():
        att.qkv.weight.copy_(mha.in_proj_weight)  # [3d, d]: Wq, Wk, Wv stacked, as ours
        att.qkv.bias.copy_(mha.in_proj_bias)
        att.out.weight.copy_(mha.out_proj.weight)
        att.out.bias.copy_(mha.out_proj.bias)
        show((att(h) - mha(h, h, h)[0]).abs().max())

(att(h) - mha(h, h, h)[0]).abs().max() = tensor(2.3842e-07, device='cuda:0')


In [23]:
blk = Block(d).to(dev)
out = blk(h)  # [4, K, d] -> [4, K, d]
show(
    out.shape,
    sum(p.numel() for p in blk.parameters()),
)

out.shape = torch.Size([4, 16, 128])
sum(p.numel() for p in blk.parameters()) = 198272


LayerNorm by hand: each token's `d` numbers to mean 0, std 1 (the learned scale and shift start at 1 and 0).


In [24]:
@lambda f: f()
def probe():
    z = h  # [4, K, d]
    mean = z.mean(-1, keepdim=True)  # [4, K, 1]: one per token
    var = z.var(-1, unbiased=False, keepdim=True)
    ours = (z - mean) / torch.sqrt(var + blk.norm1.eps)
    show(
        (ours - blk.norm1(z)).abs().max(),
        ours[0, 0].mean(),
        ours[0, 0].std(unbiased=False),
    )

(ours - blk.norm1(z)).abs().max() = tensor(7.1526e-07, device='cuda:0', grad_fn=<MaxBackward1>)
ours[0, 0].mean() = tensor(-1.4901e-08, device='cuda:0', grad_fn=<MeanBackward0>)
ours[0, 0].std(unbiased=False) = tensor(1.0000, device='cuda:0', grad_fn=<StdBackward0>)


Which output steps does one input step reach? The U-Net notebook's step 1 bumped step 0 into a kernel-5 conv: outputs 0, 1, 2 moved. Through one attention block, all 16 do.

The bump is a random vector, not +1 everywhere: +1 on all `d` numbers is a constant shift, and LayerNorm subtracts each token's mean, so attention would never see it (tried: steps 1-15 moved by 2e-5, float noise; step 0 by exactly 4 × 128 = 512, the residual carrying the bump through).


In [25]:
@lambda f: f()
def probe():
    bumped = h.clone()
    bumped[:, 0] += torch.randn(d, device=dev)  # step 0 only, a random direction
    with torch.no_grad():
        moved = (blk(bumped) - blk(h)).abs().sum(dim=(0, 2))  # [K], change at each output step
    show(
        moved.nonzero().flatten(),
        moved[0],
        moved[1:].mean(),
        moved[1:].min(),
    )

moved.nonzero().flatten() =
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15],
       device='cuda:0')
moved[0] = tensor(396.0809, device='cuda:0')
moved[1:].mean() = tensor(10.7344, device='cuda:0')
moved[1:].min() = tensor(10.6386, device='cuda:0')


### step 4 — adaLN-zero: the condition steers every block's norms, gates start at 0

The U-Net's FiLM again, aimed at the transformer block. The condition (scene + embedded `t`) predicts six vectors per block: a shift, a scale and a gate for each branch. LayerNorm's own learned scale and shift are switched off; the condition's take their place, per sample (Peebles & Xie 2023):

```
shift₁, scale₁, gate₁, shift₂, scale₂, gate₂ = Linear(SiLU(cond))      [B, d] each
x = x + gate₁ · Attn( LN(x) · (1 + scale₁) + shift₁ )
x = x + gate₂ · MLP ( LN(x) · (1 + scale₂) + shift₂ )
```

**Zero:** the `Linear` that makes the six starts at 0 weights and bias, so at init every gate is 0 and every block is exactly `x -> x`; `1 + scale` keeps the norm's scale at 1 rather than 0. A stack of these starts as the identity and each block switches itself on as training finds a use for it, as the U-Net notebook's zeroed residual block showed for one block.


In [26]:
class AdaBlock(nn.Module):
    # step 3's block with the condition's shift, scale and gate on both branches
    def __init__(self, d: int, cond_dim: int, heads: int = 4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d, elementwise_affine=False)  # the condition scales and shifts
        self.attn = Attention(d, heads)
        self.norm2 = nn.LayerNorm(d, elementwise_affine=False)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, 6 * d))
        nn.init.zeros_(self.ada[1].weight)  # zero: gates 0, block = identity at init
        nn.init.zeros_(self.ada[1].bias)

    def forward(self, x: Tensor, cond: Tensor) -> Tensor:
        # x: [B, L, d]
        # cond: [B, cond_dim]
        # out: [B, L, d]
        shift1, scale1, gate1, shift2, scale2, gate2 = self.ada(cond)[:, None].chunk(
            6, dim=-1
        )  # [B, 1, d] each
        x = x + gate1 * self.attn(self.norm1(x) * (1 + scale1) + shift1)
        return x + gate2 * self.mlp(self.norm2(x) * (1 + scale2) + shift2)

In [ ]:
ablk = AdaBlock(d, cond_dim=32).to(dev)
cond = torch.randn(4, 32, device=dev)
with torch.no_grad():
    show(
        ablk(h, cond).shape,
        (ablk(h, cond) - h).abs().max(),
    )  # identity at init: 0

ablk(h, cond).shape = torch.Size([4, 16, 128])
(ablk(h, cond) - h).abs().max() = tensor(0., device='cuda:0')


Identity at init, but not stuck: the gradient of a loss still reaches `ada` (through `gate · branch`, where `branch ≠ 0`), while the attention and MLP weights get exactly 0 (their output is multiplied by a gate of 0). So the first update only opens gates; once a gate is nonzero, its branch starts learning.


In [29]:
@lambda f: f()
def probe():
    blk_ = AdaBlock(d, cond_dim=32).to(dev)
    x = h.detach()  # h still carries step 1's graph (tok, pos); only the block's grads matter here
    target = torch.randn_like(x)
    loss = ((blk_(x, cond) - target) ** 2).mean()
    loss.backward()
    grad = lambda m: m.weight.grad.abs().max()  # noqa: E731
    show(
        grad(blk_.ada[1]),
        grad(blk_.attn.qkv),
        grad(blk_.mlp[0]),
    )

    opt = torch.optim.SGD(blk_.parameters(), lr=0.1)
    opt.step()  # one update: gates open a little
    opt.zero_grad()
    ((blk_(x, cond) - target) ** 2).mean().backward()
    show(
        'after one update',
        grad(blk_.attn.qkv),
        grad(blk_.mlp[0]),
    )

grad(blk_.ada[1]) = tensor(0.0148, device='cuda:0')
grad(blk_.attn.qkv) = tensor(0., device='cuda:0')
grad(blk_.mlp[0]) = tensor(0., device='cuda:0')
after one update
grad(blk_.attn.qkv) = tensor(0.0002, device='cuda:0')
grad(blk_.mlp[0]) = tensor(2.2981e-05, device='cuda:0')


### step 5 — the DiT velocity net

Steps 1-4 assembled, with the U-Net notebook's time embedding (its step 4) copied in:

```
cond    = MLP(top 128 ++ wrist 128 ++ state 8 ++ t_mlp(emb(t)) 64)       [B, d]   the scene and t, once
tokens  = Linear(8 → d)(x_t) + pos                                         [B, 16, d]
tokens  = AdaBlock(tokens, cond)  × N                                      steps talk, cond steers every block
out     = Linear(d → 8)( LN(tokens) · (1 + scale) + shift )                [B, 16, 8]   shift, scale from cond
```

The last `Linear` and its shift/scale start at zero too, so the untrained net predicts velocity 0 everywhere: every chunk starts as "stay where you are". Sizes are the repo's flow-matching policy's (`dim` 256, 4 layers, 4 heads, MLP 4×), values only; the structure is built here.


In [30]:
import math


def time_embedding(t: Tensor, dim: int = 32, scale: float = 1000.0) -> Tensor:
    # t: [B] in [0, 1]
    # out: [B, dim], sines then cosines at dim/2 frequencies from 1 down to 1/10000
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))  # [half]
    angles = (t * scale)[:, None] * freqs[None]  # [B, half]
    return torch.cat([angles.sin(), angles.cos()], dim=-1)

In [31]:
class DiTVelocityNet(nn.Module):
    def __init__(self, d: int = 256, layers: int = 4, heads: int = 4, t_dim: int = 64):
        super().__init__()
        self.top = nn.Sequential(make_conv(), SpatialSoftmax(16, 16))
        self.wrist = nn.Sequential(make_conv(), SpatialSoftmax(16, 16))
        self.t_dim = t_dim
        self.t_mlp = nn.Sequential(
            nn.Linear(t_dim, 4 * t_dim), nn.Mish(), nn.Linear(4 * t_dim, t_dim)
        )
        self.cond = nn.Sequential(nn.Linear(128 * 2 + 8 + t_dim, d), nn.SiLU(), nn.Linear(d, d))
        self.tok = nn.Linear(8, d)  # step 1
        self.pos = nn.Parameter(torch.randn(K, d) * 0.02)
        self.blocks = nn.ModuleList(AdaBlock(d, cond_dim=d, heads=heads) for _ in range(layers))
        self.norm_out = nn.LayerNorm(d, elementwise_affine=False)
        self.ada_out = nn.Sequential(nn.SiLU(), nn.Linear(d, 2 * d))  # shift, scale for the output
        self.out = nn.Linear(d, 8)
        for layer in (self.ada_out[1], self.out):  # zero: velocity 0 at init
            nn.init.zeros_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, top: Tensor, wrist: Tensor, state: Tensor, xt: Tensor, t: Tensor) -> Tensor:
        # top, wrist: [B, 3, 128, 128] floats in [0, 1]
        # state: [B, 8] radians
        # xt: [B, K, 8] the noisy chunk at time t, std units of action
        # t: [B] in [0, 1]
        # out: [B, K, 8] the predicted velocity, std units per unit t
        f_top = self.top((top - stats.top.mean) / stats.top.std)  # [B, 128]
        f_wrist = self.wrist((wrist - stats.wrist.mean) / stats.wrist.std)  # [B, 128]
        st = (state - stats.state.mean) / stats.state.std  # [B, 8]
        f_t = self.t_mlp(time_embedding(t, self.t_dim))  # [B, 64]
        cond = self.cond(torch.cat([f_top, f_wrist, st, f_t], dim=-1))  # [B, d]
        x = self.tok(xt) + self.pos  # [B, K, d]
        for blk in self.blocks:
            x = blk(x, cond)
        shift, scale = self.ada_out(cond)[:, None].chunk(2, dim=-1)  # [B, 1, d] each
        return self.out(self.norm_out(x) * (1 + scale) + shift)  # [B, K, 8]

In [32]:
@lambda f: f()
def probe():
    net = DiTVelocityNet().to(dev)
    i = torch.arange(4, device=dev)
    v = net(
        data.top.train[i].float() / 255,
        data.wrist.train[i].float() / 255,
        data.state.train[i],
        torch.randn(4, K, 8, device=dev),
        torch.rand(4, device=dev),
    )
    count = lambda m: sum(p.numel() for p in m.parameters())  # noqa: E731
    show(
        v.shape,
        v.abs().max(),
        count(net),
        count(net.blocks),
        count(net.top) + count(net.wrist),
    )
    show('MLP 867,202   U-Net 3,846,602')

v.shape = torch.Size([4, 16, 8])
v.abs().max() = tensor(0., device='cuda:0', grad_fn=<MaxBackward1>)
count(net) = 5394250
count(net.blocks) = 4733952
count(net.top) + count(net.wrist) = 337154
MLP 867,202   U-Net 3,846,602


### step 6 — train it (copied from `flow_matching_unet.ipynb` step 6, net swapped; ~5 min the first time, cached after)

The same batches, loss, Euler sampler, val error, and cached `train` / `load`; only the net is `DiTVelocityNet`. Same seed, steps, lr, batch 64. At 10k: MLP val arm 0.0174, gripper 0.0909; U-Net 0.0117, 0.0351.


In [33]:
from typing import Literal, Protocol

import torch.nn.functional as F


def random_shift(x: Tensor, pad: int = 4) -> Tensor:
    # as bc_2 step 30
    # x: [B, 3, H, W]
    # out: [B, 3, H, W], the whole batch moved by up to pad px each way, edge pixels repeated
    B, C, H, W = x.shape
    x = F.pad(x, (pad, pad, pad, pad), mode='replicate')  # [B, 3, H + 2p, W + 2p]
    dy, dx = (int(d) for d in torch.randint(0, 2 * pad + 1, (2,)))
    return x[:, :, dy : dy + H, dx : dx + W]


class Batch(Protocol):  # what get_batch returns: a namespace with these fields
    top: Tensor  # [B, 3, 128, 128] floats in [0, 1]
    wrist: Tensor  # [B, 3, 128, 128] floats in [0, 1]
    state: Tensor  # [B, 8] radians
    chunks: Tensor  # [B, K, 8] radians
    pad: Tensor  # [B, K] bool, True past the demo's end


def get_batch(part: Literal['train', 'val'], i: Tensor) -> Batch:
    # rows i of one part, every field at once; images to floats in [0, 1] only now
    def take(x) -> Tensor:  # x: a train/val pair from step 9's split
        return (x.train if part == 'train' else x.val)[i]

    return namespace(
        top=take(data.top).float() / 255,  # [B, 3, 128, 128]
        wrist=take(data.wrist).float() / 255,  # [B, 3, 128, 128]
        state=take(data.state),  # [B, 8]
        chunks=take(data.chunks),  # [B, K, 8] radians
        pad=take(data.pad),  # [B, K] bool
    )

In [34]:
def flow_loss(net: DiTVelocityNet, b: Batch, shift: int = 4) -> Tensor:
    # out: [], the toy's loss on chunks: MSE between predicted and true speed, padded steps skipped
    x1 = (b.chunks - stats.action.mean) / stats.action.std  # [B, K, 8], the demo chunk, std units
    x0 = torch.randn_like(x1)  # [B, K, 8], noise
    t = torch.rand(len(x1), device=dev)  # [B]
    xt = (1 - t[:, None, None]) * x0 + t[:, None, None] * x1  # [B, K, 8], the point at time t
    v = net(random_shift(b.top, shift), random_shift(b.wrist, shift), b.state, xt, t)
    err = (v - (x1 - x0)) ** 2  # [B, K, 8], MSE on purpose: its best guess is the mean speed
    keep = (~b.pad)[..., None].float()  # [B, K, 1], 0 on padded steps
    return (err * keep).sum() / (keep.sum() * 8)  # over real values only

In [35]:
torch.manual_seed(0)
b = get_batch('train', torch.randint(0, len(data.state.train), (64,), device=dev))
with torch.no_grad():
    loss = flow_loss(
        DiTVelocityNet().to(dev), b
    )  # untrained: v = 0 exactly (zero output layer), so about E|x1 - x0|² = 1 + 1 = 2
show(
    b.pad.float().mean(),
    loss,
)

b.pad.float().mean() = tensor(0.0342, device='cuda:0')
loss = tensor(2.0585, device='cuda:0')


In [36]:
@torch.no_grad()
def sample(
    net: DiTVelocityNet, top: Tensor, wrist: Tensor, state: Tensor, steps: int = 10
) -> Tensor:
    # the toy's sampler (flow_matching.ipynb step 4), per observation
    # out: [B, K, 8] radians, one sampled plan per observation
    x = torch.randn(len(state), K, 8, device=dev)  # [B, K, 8], fresh noise
    for k in range(steps):
        t = torch.full((len(state),), k / steps, device=dev)  # [B]
        x = x + net(top, wrist, state, x, t) / steps  # Euler: x += speed * dt
    return x * stats.action.std + stats.action.mean  # std units -> radians


@torch.no_grad()
def val_error(net: DiTVelocityNet) -> Tensor:
    # out: [8], mean |error| of one sampled plan's FIRST action per val frame, radians,
    # comparable with BC and ACT. 256 frames at a time, never shifted
    n = len(data.state.val)
    firsts = []
    for j in range(0, n, 256):
        b = get_batch('val', torch.arange(j, min(j + 256, n), device=dev))
        firsts.append(sample(net, b.top, b.wrist, b.state)[:, 0])  # [<=256, 8]
    return (torch.cat(firsts) - data.chunks.val[:, 0]).abs().mean(0)

In [37]:
@lambda f: f()
def probe():
    # seconds per training step, to size the train cell (50 steps after 5 warm-up)
    import time

    net = DiTVelocityNet().to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3)
    for step in range(55):
        if step == 5:
            torch.cuda.synchronize()
            start = time.time()
        b = get_batch('train', torch.randint(0, len(data.state.train), (64,), device=dev))
        loss = flow_loss(net, b)
        opt.zero_grad()
        loss.backward()
        opt.step()
    torch.cuda.synchronize()
    per_step = (time.time() - start) / 50
    show(
        per_step,
        f'10k steps ≈ {per_step * 10000 / 60:.1f} min of training',
    )

per_step = 0.026642370223999023
10k steps ≈ 4.4 min of training


In [38]:
@cache
def train(seed: int = 0, steps: int = 10000, lr: float = 1e-3) -> namespace:
    torch.manual_seed(seed)
    net = DiTVelocityNet().to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=lr)
    err = torch.zeros(8)
    for step in range(steps + 1):
        b = get_batch('train', torch.randint(0, len(data.state.train), (64,), device=dev))
        loss = flow_loss(net, b)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 2000 == 0:
            net.eval()
            err = val_error(net).cpu()
            net.train()
            print(
                f'{step:5d}  loss {loss.item():.3f}  val arm {err[:7].mean():.4f}  gripper {err[7]:.4f}'
            )
    # weights on the CPU, so the cache holds no GPU tensors
    return namespace(weights={k: v.cpu() for k, v in net.state_dict().items()}, err=err)

train: not cached, will compute


In [39]:
def load(train_steps: int = 10000, log: bool = False) -> DiTVelocityNet:
    # the net train(steps=train_steps) made, from its cache; log: replay its training log
    net = DiTVelocityNet().to(dev)
    net.load_state_dict((train if log else train.quiet)(steps=train_steps).weights)
    return net.eval()


vnet = load(log=True)

    0  loss 2.059  val arm 0.1649  gripper 0.5466
 2000  loss 0.080  val arm 0.0129  gripper 0.0564
 4000  loss 0.075  val arm 0.0127  gripper 0.0452
 6000  loss 0.080  val arm 0.0111  gripper 0.0410
 8000  loss 0.060  val arm 0.0096  gripper 0.0350
10000  loss 0.056  val arm 0.0093  gripper 0.0336


**Result (step 6):** at 10k steps val arm 0.0093 (U-Net 0.0117, MLP 0.0174), gripper 0.0336 (U-Net 0.0351, MLP 0.0909); loss 0.056 (0.061, 0.188). The lowest arm error of the three, gripper level with the U-Net.

### step 7 — the sim, the rollout and both tests (copied from `flow_matching_unet.ipynb` step 7)

At 10k, same start (8 / 16 steps per plan): MLP 5/20, 17/20 (9 right / 8 left); U-Net 20/20 (14 L / 6 R), 20/20 (all left). Spread: MLP 7/20, 15/20; U-Net 19/20, 18/20.


In [40]:
import logging
import os

# render without a window; must be set before mujoco loads
os.environ.setdefault('MUJOCO_GL', 'egl')

logging.disable(logging.WARNING)  # robosuite warns while importing; mute that
import robosuite

logging.disable(logging.NOTSET)
logging.getLogger('robosuite_logs').setLevel(logging.ERROR)
from dream_robot.core.registry import make_env
from dream_robot.core.schema import Observation
from dream_robot.sims.robosuite.tasks.pick_place_two_bins.env import PickPlaceTwoBins

env: PickPlaceTwoBins = make_env(
    'robosuite/pick_place_two_bins_fixed'
)  # cube always at the centre, either bin wins

In [41]:
def frame(im: np.ndarray) -> Tensor:
    # im: [H, W, 3] uint8 from the sim
    # out: [1, 3, H, W] floats in [0, 1] on the GPU
    return torch.from_numpy(im).permute(2, 0, 1)[None].to(dev).float() / 255


def predict(net: DiTVelocityNet, obs: Observation) -> NDArray[np.float32]:
    # out: [K, 8], one sampled plan from this observation; fresh noise each call
    st = torch.from_numpy(obs.state)[None].to(dev)  # [1, 8]
    plan = sample(net, frame(obs.images['top']), frame(obs.images['wrist']), st)[0].cpu().numpy()
    plan[:, 7] = np.clip(plan[:, 7], 0.0, 1.0)  # gripper commands must stay in [0, 1]
    return plan

In [42]:
from typing import NamedTuple

from dream_robot.core.video import observation_panel


class Outcome(NamedTuple):  # how one rollout ended
    success: bool
    where: str  # where the cube ended: 'left' / 'right' / 'between' / 'other'


def rollout(net: DiTVelocityNet, seed: int, n_exec: int = 8, frames: list | None = None) -> Outcome:
    # sample a plan, play its first n_exec actions without looking, then sample again;
    # frames: if a list, one video frame per step is appended to it (wide view + both cameras)
    obs = env.reset(seed=seed)
    plan = []  # actions still to play from the last plan
    success = False
    for _ in range(600):
        if frames is not None:
            frames.append(observation_panel(env.render(), obs.images))
        if not plan:
            plan = list(predict(net, obs)[:n_exec])  # n_exec x [8]
        r = env.step(plan.pop(0))
        obs = r.observation
        if r.terminated or r.truncated:
            success = r.success
            break
    cube = env.cube_pos
    b = env.which_bin(cube)  # index into env.bin_names, or None
    where = (
        env.bin_names[b] if b is not None else ('between' if env.between_bins(cube) else 'other')
    )
    return Outcome(success, where)

In [43]:
torch.manual_seed(0)
ok, where = rollout(vnet, 1000)
show(
    ok,
    where,
)

ok = True
where = 'left'


In [44]:
from collections import Counter


def score(results: list[Outcome]) -> str:
    # 'wins/n   {where: count}'
    wins = sum(r.success for r in results)
    return f'{wins:2d}/{len(results)}   {dict(Counter(r.where for r in results))}'

In [45]:
@cache
def same_start(train_steps: int = 10000, n_exec: int = 8) -> list[Outcome]:
    # does it use BOTH bins? One scene (env seed 1000) played 20 times; only the net's noise
    # differs (torch seed k). The expert flips a coin here, so the target is ~10 left, ~10 right.
    # BC and ACT are deterministic: the same bin 20/20 by construction
    net = load(train_steps)
    results = []
    for k in range(20):
        torch.manual_seed(k)
        results.append(rollout(net, 1000, n_exec))
    print(f'same start, {train_steps} steps, n_exec {n_exec}: {score(results)}')
    return results

same_start: not cached, will compute


In [46]:
@cache
def spread(train_steps: int = 10000, n_exec: int = 8) -> list[Outcome]:
    # does it WORK? The standard eval, as for BC and ACT: 20 different scenes (env seeds
    # 1000-1019), one rollout each, one torch seed for the whole run. The cube is always at the
    # centre in T2-fixed, so the scenes differ only in the start's small jitter
    net = load(train_steps)
    torch.manual_seed(0)
    results = [rollout(net, seed, n_exec) for seed in range(1000, 1020)]
    print(f'spread,     {train_steps} steps, n_exec {n_exec}: {score(results)}')
    return results

spread: not cached, will compute


In [48]:
# both tests at both ways of playing the plans (~10 min the first time, cached after)
for n_exec in [8, 16]:
    same_start(10000, n_exec)
    spread(10000, n_exec)

same start, 10000 steps, n_exec 8: 20/20   {'left': 12, 'right': 8}
spread,     10000 steps, n_exec 8: 20/20   {'left': 13, 'right': 7}
same start, 10000 steps, n_exec 16: 20/20   {'left': 15, 'right': 5}
spread,     10000 steps, n_exec 16: 20/20   {'left': 12, 'right': 8}


**Result (step 7)**, 10k steps, one seed each:

| net | params | val arm / gripper | same start, 8 | same start, 16 | spread, 8 | spread, 16 |
|---|---|---|---|---|---|---|
| MLP | 867k | 0.0174 / 0.0909 | 5/20 (between 15) | 17/20 (9 R / 8 L) | 7/20 | 15/20 |
| U-Net | 3.85M | 0.0117 / 0.0351 | 20/20 (14 L / 6 R) | 20/20 (20 L) | 19/20 | 18/20 |
| **DiT** | 5.39M | **0.0093 / 0.0336** | **20/20 (12 L / 8 R)** | **20/20 (15 L / 5 R)** | **20/20** | **20/20** |

- **The DiT is the only one at 80/80**: no slipped grasp in any test, at either way of playing the plans.
- **And it keeps both bins**: 12/8 and 15/5 from one scene, where the U-Net went 20/0 at 16. Still left-leaning (27 L / 13 R over both; a fair coin gives ≥27 of 40 about 2% of the time), as every net here has been once it grasps well.
- Bigger than both (5.39M), so size and architecture aren't separated; one seed each.

## wrap-up — the velocity-net series (MLP → U-Net → DiT)

Same data, loss, sampler and tests in all three notebooks; only the velocity net changed. 10k steps, seed 0:

| net | params | val arm / gripper | same start, 8 | same start, 16 | spread, 8 / 16 |
|---|---|---|---|---|---|
| MLP (`flow_matching_namespace.ipynb`) | 867k | 0.0174 / 0.0909 | 5/20 (between 15) | 17/20 (9 R / 8 L) | 7, 15 |
| U-Net (`flow_matching_unet.ipynb`) | 3.85M | 0.0117 / 0.0351 | 20/20 (14 L / 6 R) | 20/20 (20 L) | 19, 18 |
| DiT (this notebook) | 5.39M | 0.0093 / 0.0336 | 20/20 (12 L / 8 R) | 20/20 (15 L / 5 R) | 20, 20 |

**What the DiT buys:** every step reads every other step in one block (step 3: one bumped step moved all 16 outputs, the U-Net's conv moved 3), and the condition steers every block's norms through adaLN-zero, each block starting as the identity and switching itself on (step 4: attention and MLP gradients exactly 0 until the gates open). Result: the only net at 80/80, and both bins kept from one scene.

**Where it fails / what's unknown:**
- It still leans left (27 L / 13 R over both same-start tests; ≥27 of 40 from a fair coin ≈ 2%). Every net here leans left once it grasps well; whether that is flow matching, these 100 demos, or the start scene is open.
- Size and architecture are not separated (867k → 3.85M → 5.39M).

**Pending: the seed-1 comparison.** Every number above is one seed (seed 0). Train all three at seed 1 (10k steps each, ~5 min; the MLP also at 30k) and rerun both tests, to see whether the ranking and the left lean hold.

**Next after that:** where the left lean comes from: sample 1000 plans from the one start scene and count which bin each heads for, before any rollout.